# 07 — Registry Autorun Key Persistence

| Field | Value |
|-------|-------|
| **MITRE ATT&CK ID** | T1547.001 — Boot or Logon Autostart Execution: Registry Run Keys |
| **Tactic** | Persistence |
| **Difficulty** | Intermediate |
| **Estimated Time** | 35 minutes |

## Threat Context: Emotet — Registry Run Key Persistence

Emotet, one of the most prolific banking trojans turned malware distribution platform, extensively used Windows Registry Run keys for persistence. After initial infection (typically via a malicious Word macro), Emotet copied itself to `%AppData%` and created a Run key entry to execute on every user login. At its peak (2018-2021), Emotet infected over 1.6 million machines worldwide, and registry persistence was central to its survival across reboots.

The Windows Registry is a hierarchical database that stores OS and application configuration. The `Run` and `RunOnce` keys under `HKCU\Software\Microsoft\Windows\CurrentVersion\` automatically execute specified programs at user login — a feature attackers have abused since the early 2000s.

## What You'll Understand After This

- How **Registry Run keys** provide persistence by executing programs at user login or system boot
- The full set of **autorun registry locations** that attackers target
- How to **detect and monitor** registry-based persistence using Sysmon, EDR, and forensic tools

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Environment setup and imports

import os
import json
import platform
from collections import OrderedDict

# Simulated registry for cross-platform demonstration
# On Windows, winreg would be used; here we simulate the structure
SIMULATED_REGISTRY = {}

print("[+] Imports loaded successfully")
print(f"[i] Running on: {platform.system()}")
print("[i] This notebook SIMULATES registry operations — no actual registry is modified.")
if platform.system() != "Windows":
    print("[i] Using simulated registry structure (non-Windows platform detected).")

### Step 1 — Map Autorun Registry Locations

Windows has many registry locations that can trigger program execution. Attackers choose locations based on privilege level and stealth requirements.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — autorun registry location mapping

autorun_locations = [
    {
        "path": r"HKCU\Software\Microsoft\Windows\CurrentVersion\Run",
        "privilege": "User",
        "trigger": "User logon",
        "attacker_use": "Most common — no admin needed",
        "risk_level": "HIGH"
    },
    {
        "path": r"HKLM\Software\Microsoft\Windows\CurrentVersion\Run",
        "privilege": "Admin",
        "trigger": "Any user logon",
        "attacker_use": "Runs for ALL users — requires elevation",
        "risk_level": "HIGH"
    },
    {
        "path": r"HKCU\Software\Microsoft\Windows\CurrentVersion\RunOnce",
        "privilege": "User",
        "trigger": "Next logon only (then deleted)",
        "attacker_use": "One-shot execution, self-cleaning",
        "risk_level": "MEDIUM"
    },
    {
        "path": r"HKLM\Software\Microsoft\Windows NT\CurrentVersion\Winlogon\Shell",
        "privilege": "Admin",
        "trigger": "Shell initialization",
        "attacker_use": "Replace explorer.exe — drastic but effective",
        "risk_level": "CRITICAL"
    },
    {
        "path": r"HKLM\Software\Microsoft\Windows NT\CurrentVersion\Winlogon\Userinit",
        "privilege": "Admin",
        "trigger": "After user authentication",
        "attacker_use": "Append payload to userinit chain",
        "risk_level": "HIGH"
    },
    {
        "path": r"HKCU\Software\Microsoft\Windows\CurrentVersion\Explorer\User Shell Folders",
        "privilege": "User",
        "trigger": "Explorer startup",
        "attacker_use": "Redirect Startup folder to attacker-controlled path",
        "risk_level": "MEDIUM"
    },
]

print("[+] Windows Autorun Registry Locations:")
print(f"{'Location':<65} {'Priv':<7} {'Risk'}")
print("-" * 85)
for loc in autorun_locations:
    print(f"{loc['path']:<65} {loc['privilege']:<7} {loc['risk_level']}")

### Step 2 — Simulate Registry Modification

We simulate what happens when malware writes a Run key. On a real Windows system, this would use the `winreg` module. Here we use an in-memory dictionary.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — simulated registry operations

def sim_reg_write(hive_path, value_name, value_data):
    """Simulate writing a registry value."""
    if hive_path not in SIMULATED_REGISTRY:
        SIMULATED_REGISTRY[hive_path] = {}
    SIMULATED_REGISTRY[hive_path][value_name] = value_data
    return True

def sim_reg_read(hive_path):
    """Simulate reading all values from a registry key."""
    return dict(SIMULATED_REGISTRY.get(hive_path, {}))

def sim_reg_delete(hive_path, value_name):
    """Simulate deleting a registry value."""
    if hive_path in SIMULATED_REGISTRY and value_name in SIMULATED_REGISTRY[hive_path]:
        removed = SIMULATED_REGISTRY[hive_path].pop(value_name)
        return removed
    return None

# Simulate Emotet-style persistence
run_key = r"HKCU\Software\Microsoft\Windows\CurrentVersion\Run"

# Legitimate entries (pre-existing)
sim_reg_write(run_key, "SecurityHealth", r"C:\Windows\System32\SecurityHealthSystray.exe")
sim_reg_write(run_key, "OneDrive", r'"C:\Users\user\AppData\Local\Microsoft\OneDrive\OneDrive.exe" /background')

# Simulated malicious entry (Emotet-style)
sim_reg_write(run_key, "WindowsOptimizer",
              r'"C:\Users\user\AppData\Roaming\mshelper\svchost.exe" -q')

print("[+] Simulated Run key contents:")
entries = sim_reg_read(run_key)
for name, data in entries.items():
    suspicious = "<-- SUSPICIOUS" if "mshelper" in data else ""
    print(f"  {name}: {data} {suspicious}")

print("\n[i] Attacker's entry mimics a legitimate service name ('WindowsOptimizer')")
print("[i] The payload path uses AppData (no admin needed) with a legitimate-sounding name")

### Step 3 — Detection: Baseline and Diff

A key detection strategy is maintaining a baseline of expected Run key entries and alerting on new additions.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — registry baseline and diff detection

# Baseline: known-good entries
baseline = {
    "SecurityHealth": r"C:\Windows\System32\SecurityHealthSystray.exe",
    "OneDrive": r'"C:\Users\user\AppData\Local\Microsoft\OneDrive\OneDrive.exe" /background',
}

# Current state (includes malicious entry)
current = sim_reg_read(run_key)

# Diff detection
new_entries = {k: v for k, v in current.items() if k not in baseline}
modified_entries = {k: v for k, v in current.items()
                    if k in baseline and baseline[k] != v}
removed_entries = {k: v for k, v in baseline.items() if k not in current}

detection_results = []

print("[+] Registry Baseline Diff Analysis:")
print(f"    Baseline entries: {len(baseline)}")
print(f"    Current entries:  {len(current)}")

if new_entries:
    print(f"\n  [ALERT] {len(new_entries)} NEW entries detected:")
    for name, data in new_entries.items():
        print(f"    + {name}: {data}")
        detection_results.append({"type": "NEW", "name": name, "data": data})

if modified_entries:
    print(f"\n  [ALERT] {len(modified_entries)} MODIFIED entries:")
    for name, data in modified_entries.items():
        print(f"    ~ {name}: {baseline[name]} -> {data}")

if not new_entries and not modified_entries:
    print("  [OK] No deviations from baseline")

print(f"\n[+] Detection results: {len(detection_results)} anomalies found")

### Visualization — Autorun Location Tree

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.axis("off")
ax.set_title("Windows Registry Autorun Locations", fontsize=14, fontweight="bold")

# Root nodes
root_boxes = [
    (1, 7, "HKCU", "#3498db"),
    (7, 7, "HKLM", "#e74c3c"),
]

# HKCU children
hkcu_items = [
    (0.2, 5.5, "Run\n(User logon)", "#2ecc71"),
    (0.2, 4.0, "RunOnce\n(Next logon)", "#f1c40f"),
    (0.2, 2.5, "User Shell\nFolders", "#f1c40f"),
]

# HKLM children
hklm_items = [
    (6.2, 5.5, "Run\n(All users)", "#e74c3c"),
    (6.2, 4.0, "Winlogon\nShell", "#c0392b"),
    (6.2, 2.5, "Winlogon\nUserinit", "#e74c3c"),
]

for x, y, label, color in root_boxes:
    rect = mpatches.FancyBboxPatch((x, y - 0.3), 2.5, 0.7, boxstyle="round,pad=0.1",
                                   facecolor=color, edgecolor="white")
    ax.add_patch(rect)
    ax.text(x + 1.25, y, label, ha="center", va="center", color="white",
            fontsize=12, fontweight="bold")

for items, root_x in [(hkcu_items, 2.25), (hklm_items, 8.25)]:
    for x, y, label, color in items:
        rect = mpatches.FancyBboxPatch((x, y - 0.4), 3.5, 0.9, boxstyle="round,pad=0.1",
                                       facecolor=color, edgecolor="white", alpha=0.8)
        ax.add_patch(rect)
        ax.text(x + 1.75, y, label, ha="center", va="center", color="white",
                fontsize=9, fontweight="bold")
        ax.plot([root_x, x + 1.75], [6.7, y + 0.5], color="gray", linewidth=1.5, alpha=0.5)

# Legend
legend_items = [
    mpatches.Patch(color="#2ecc71", label="User-level (no admin)"),
    mpatches.Patch(color="#f1c40f", label="User-level (less common)"),
    mpatches.Patch(color="#e74c3c", label="Admin required"),
    mpatches.Patch(color="#c0392b", label="Critical — system shell"),
]
ax.legend(handles=legend_items, loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()
print("[+] Visualization complete")

## Defender's Perspective — Indicators of Compromise

| Indicator | Type | Detection Method |
|-----------|------|------------------|
| New Run/RunOnce key value created | Registry | Sysmon Event ID 12/13, EDR registry monitoring |
| Run key pointing to `%AppData%` or `%Temp%` path | Registry | Registry baseline diff, YARA rules |
| Executable with legitimate-sounding name in unusual path | File | File reputation, hash lookup, static analysis |
| Registry modification by non-interactive process | Behavioral | Process lineage tracking (parent-child anomaly) |
| Run key value referencing encoded or obfuscated command | Registry | Regex-based registry value scanning |

## Article Seed

**Title:** *The Registry Run Key: Persistence's Oldest Trick Still Works*

**Opening Paragraph:**
Twenty years after the first malware abused Windows Registry Run keys, the technique remains one of the most commonly observed persistence mechanisms in enterprise breaches. From Emotet to Cobalt Strike, threat actors continue to rely on this deceptively simple approach because it requires no admin privileges, survives reboots, and hides among legitimate autostart entries.

**Section Headers:**
1. Registry Run Keys: A Deep Dive Into Windows Autostart Mechanisms
2. How Emotet Leveraged Registry Persistence at Scale
3. Detection Engineering: Sysmon Rules and Baseline Monitoring
4. Prevention: Group Policy and Registry Access Control

**Medium Tags:** `cybersecurity`, `persistence`, `windows-registry`, `mitre-attack`, `malware-analysis`

In [ ]:
# Self-check assertions

# 1. Verify autorun locations were catalogued
assert len(autorun_locations) >= 5, "Should catalogue at least 5 autorun locations"

# 2. Verify baseline diff detected the malicious entry
assert len(detection_results) == 1, f"Expected 1 anomaly, got {len(detection_results)}"
assert detection_results[0]["name"] == "WindowsOptimizer"

# 3. Verify simulated registry contains expected entries
assert len(sim_reg_read(run_key)) == 3, "Run key should have 3 entries (2 legit + 1 malicious)"

print("[+] All self-check assertions passed!")